In [1]:
# 데이터 생성 : csv파일을 불러옴
import pandas as pd

df = pd.read_csv('../data/csv/sentiment_data_long.csv')


In [3]:
# 독립, 종속변수 분리
X = df['sentence']
y = df['label']


In [4]:
# 훈련, 테스트 분리 
DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE: ]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE: ]

In [5]:
# 형태소 분리 시 모든 형태소를 포함
# 분리한 단어들을 합침
from konlpy.tag import Okt

def get_preprocessing(sentence):
	okt = Okt()
	result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단, 원형으로
	words = [word for word, pos in result]
	return " ".join(words)

# X_train과 X_test의 있는 문장들을 get_preprocessing을 적용
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)

In [7]:
# 벡터화 
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode="int",
	output_sequence_length=10
)
vectorize_layer.adapt(X_train.tolist())

In [10]:
import tensorflow as tf

# 모델 설계
model = models.Sequential([
	layers.Input((1, ), dtype=tf.string),
	vectorize_layer, 
	layers.Embedding(input_dim=1000, output_dim=64),
	layers.LSTM(32),
	layers.Dense(1, activation='sigmoid')#출력층
])

In [12]:
model.compile(
	optimizer='adam', 
	loss='binary_crossentropy', 
	metrics=['accuracy']
) 

In [15]:
history = model.fit(X_train.values, y_train, epochs=50, verbose=1)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 9.4076e-05
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 9.0956e-05
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 8.7975e-05
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 8.5142e-05
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 8.2443e-05
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 7.9871e-05
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 7.7414e-05
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 7.5066e-05
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 7.2815e-05
Epoch 10/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 7.0663e-05
Epoch 11/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 6.8603e-05
Epoch 12/50
25/25 ━━━━━━━━━━━━

In [16]:
# 평가
_, acc = model.evaluate(X_test.values, y_test)
print(f'정확도 : {acc}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 3.6392e-05 
정확도 : 1.0


In [ ]:
# 예측
# "너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요"
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요
import numpy as np
sentences = [
	"너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요",
	"지루한 장면이 반복되어서 스토리가 너무 개연성이 없고 결말이 허무하기 짝이 없다"
]
# 학습 데이터가 긍정=> 부정, 부정=>긍정인 데이터들이어서 
# 긍정 문구가 있으면 부정으로 될거라고 예측
# 부정문구가 있으면 긍정으로 될거라고 예측

predictions = model.predict(np.array(sentences,dtype=object))
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step


array([[0.99918795],
       [0.9939022 ]], dtype=float32)